# Aula 28 — Avaliação e Simulação Integrada do Módulo 3 (Aqua Index)

**UNIFEI • ECAA08 Automática • Grupo 2**

Este notebook executa uma simulação integrada *end-to-end* das estruturas em árvore estudadas nas **Aulas 19 a 28** sobre o caso de estudo conceitual **Aqua Index** (envasadora pneumática Setores 000–500, *takt time* nominal `3,6 s`), validando 4 cenários operacionais com `assert`s automatizados.

> **Nota conceitual (`PRODUCT.md`):** Simulação didática em Python 3.13 + NumPy; não representa planta física construída nem certificação SIL. A emergência `HS-004` real é *hardwired*.

In [ ]:
import heapq
import math
from collections import deque
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

print(f"Ambiente Python/NumPy pronto (matplotlib={'sim' if HAS_MPL else 'não'})")

In [ ]:
# ============================================================================
# 1. AULA 19 (MST) + AULA 23 (RSTP 802.1w) + AULA 20 (HUFFMAN)
# ============================================================================

MST_NODES = ["PLC-001", "MESA-001", "E1-DISP", "E2-ENV", "E3-TAMP", "E4-SEL", "E5-EJEC", "EST-501"]
MST_EDGES = [
    ("E5-EJEC", "EST-501", 1.1), ("E1-DISP", "E2-ENV", 1.2), ("E2-ENV", "E3-TAMP", 1.3),
    ("E3-TAMP", "E4-SEL", 1.4), ("E4-SEL", "E5-EJEC", 1.5), ("MESA-001", "E1-DISP", 1.6),
    ("MESA-001", "E2-ENV", 1.7), ("PLC-001", "E1-DISP", 1.8), ("MESA-001", "E3-TAMP", 1.9),
    ("PLC-001", "MESA-001", 2.0), ("MESA-001", "E4-SEL", 2.1), ("MESA-001", "E5-EJEC", 2.2),
    ("E5-EJEC", "E1-DISP", 2.6), ("PLC-001", "EST-501", 3.4)
]

def kruskal_mst(nodes, edges):
    parent = {n: n for n in nodes}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    selected = []
    for u, v, w in sorted(edges, key=lambda item: item[2]):
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[rv] = ru
            selected.append((u, v, w))
    total_w = round(sum(w for _, _, w in selected), 2)
    full_w = round(sum(w for _, _, w in edges), 2)
    return {"edges": selected, "total_m": total_w, "full_m": full_w, "reduction_pct": round((1 - total_w / full_w) * 100, 1)}

RSTP_SWITCHES = {
    "SW-SCADA": 4096,
    "SW-PLC": 8192,
    "SW-IO-IN": 32768,
    "SW-IO-PROC": 32768,
    "SW-IO-OUT": 32768,
}
RSTP_LINKS = [
    ("L1", "SW-SCADA", "SW-PLC", 2000),
    ("L2", "SW-SCADA", "SW-IO-OUT", 20000),
    ("L3", "SW-PLC", "SW-IO-IN", 20000),
    ("L4", "SW-IO-OUT", "SW-IO-PROC", 20000),
    ("L5", "SW-IO-IN", "SW-IO-PROC", 20000),
    ("L6", "SW-PLC", "SW-IO-PROC", 200000),
]

def run_rstp(backbone_down=False, scada_down=False, takt_s=3.6):
    active_nodes = {k: v for k, v in RSTP_SWITCHES.items() if not (scada_down and k == "SW-SCADA")}
    root = min(active_nodes, key=lambda s: (active_nodes[s], s))
    usable = []
    for lid, u, v, cost in RSTP_LINKS:
        if u not in active_nodes or v not in active_nodes:
            continue
        if backbone_down and lid == "L1":
            continue
        usable.append((lid, u, v, cost))
    dist = {root: 0}
    parent_link = {}
    pq = [(0, root)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, math.inf):
            continue
        for lid, a, b, cost in usable:
            if a == u or b == u:
                v = b if a == u else a
                nd = d + cost
                if nd < dist.get(v, math.inf):
                    dist[v] = nd
                    parent_link[v] = lid
                    heapq.heappush(pq, (nd, v))
    forwarding = set(parent_link.values())
    discarding = {lid for lid, _, _, _ in usable if lid not in forwarding}
    rstp_conv_s = 0.0 if not (backbone_down or scada_down) else 0.15
    stp_conv_s = 0.0 if not (backbone_down or scada_down) else 36.0
    return {
        "root": root,
        "forwarding": sorted(forwarding),
        "discarding": sorted(discarding),
        "plc_cost": dist.get("SW-PLC", 0),
        "lost_cycles_rstp": math.floor(rstp_conv_s / takt_s),
        "lost_cycles_stp": math.floor(stp_conv_s / takt_s),
    }

HUFFMAN_CODES = {"00000": "1", "10000": "00", "00100": "010", "00010": "0111", "01000": "01101", "00001": "01100"}
HUFFMAN_PROBS = {"00000": 0.88, "10000": 0.055, "00100": 0.03, "00010": 0.02, "01000": 0.012, "00001": 0.003}

mst_res = kruskal_mst(MST_NODES, MST_EDGES)
assert mst_res["total_m"] == 9.9 and mst_res["reduction_pct"] == 61.6
avg_len = round(sum(HUFFMAN_PROBS[k] * len(HUFFMAN_CODES[k]) for k in HUFFMAN_PROBS), 4)
assert avg_len == 1.235
assert run_rstp()["root"] == "SW-SCADA" and run_rstp(backbone_down=True)["lost_cycles_rstp"] == 0
assert run_rstp(backbone_down=True)["lost_cycles_stp"] == 10
print("Aulas 19, 20 e 23 validadas:", mst_res["total_m"], "m | L =", avg_len, "bits | RSTP 0 ciclos perdidos")

In [ ]:
# ============================================================================
# 2. AULA 21 (BDT + AVL), AULA 24 (PETRI PRÉVIA), AULA 25 (BT), AULA 26 (KD-TREE), AULA 22 (RRT), AULA 27 (CART)
# ============================================================================

def evaluate_bdt(hs004=False, r06=False, req_giro=True, zsc401=True, req_prensa=False, tit401=186.0, req_braco=False, pit301=True, pos_disp=False, zs102=True):
    if hs004:
        return "TRIP_EMERGENCIA"
    if r06:
        return "TRAVA_SENSOR_MAGNETICO"
    if req_giro and not zsc401:
        return "BLOQUEIO_COLISAO_MESA"
    if req_prensa and tit401 < 180.0:
        return "HOLD_ESTAGIO_4"
    if req_braco and not pit301:
        return "HOLD_ESTAGIO_3"
    if pos_disp and not zs102:
        return "ALARME_REABASTECIMENTO"
    return "OPERACAO_NORMAL"

def petri_bounded_check(k_max=3):
    # Simulação BFS de espaço de estados com lugar complementar K_max em EST-500 e exclusão mútua Mesa x Prensa
    visited = set()
    q = deque([(1, 0, 0, k_max)])  # (Mesa_Livre, Prensa_Ativa, Est500_Ocup, Est500_Comp)
    while q:
        m, p, est, comp = q.popleft()
        st = (m, p, est, comp)
        if st in visited:
            continue
        visited.add(st)
        assert not (m == 1 and p == 1), "Violação de exclusão mútua Mesa x Prensa!"
        assert est + comp == k_max, "Invariante de capacidade da esteira violada!"
        if m == 1:
            q.append((0, 1, est, comp))
        if p == 1 and comp > 0:
            q.append((1, 0, est + 1, comp - 1))
        if est > 0:
            q.append((m, p, est - 1, comp + 1))
    return {"states": len(visited), "bounded": True, "deadlocks": 0, "mutual_exclusion": True}

def tick_behavior_tree(hs004=False, zs102=True, pit301=True, tit401=186.0, retry_count=0):
    if hs004:
        return {"root": "SUCCESS", "branch": "Seguranca_Global", "action": "Parada_Segura (M-001=0, HT-401=0, IL-002=1)"}
    if not zs102:
        return {"root": "RUNNING", "branch": "Recuperacao_ZS102", "action": "Pausar_e_Solicitar_MAG100"}
    if not pit301:
        if retry_count < 2:
            return {"root": "RUNNING", "branch": "Recuperacao_PIT301", "action": f"Retry_VAC301 ({retry_count + 1}/2)"}
        return {"root": "FAILURE", "branch": "Recuperacao_PIT301", "action": "Alarme_R03_Vacuo_Persistente"}
    if tit401 < 180.0:
        return {"root": "RUNNING", "branch": "Aquecimento_HT401", "action": f"Aquecer_HT401 ({tit401:.1f}C < 180C)"}
    return {"root": "SUCCESS", "branch": "Ciclo_Produtivo_Parallel", "action": "E1..E5_Selagem_e_Mesa_72_Graus"}

def kdtree_nearest_2d(points, query):
    best = min(points, key=lambda p: (p[0] - query[0]) ** 2 + (p[1] - query[1]) ** 2)
    return best, round(math.hypot(best[0] - query[0], best[1] - query[1]), 3)

def cart_predict_cycle(tit401, zs102, pit301, takt_s=3.6):
    if not zs102 or takt_s > 4.8:
        return "PARADA_ABASTECIMENTO_ATRASO"
    if not pit301:
        return "FALHA_TAMPA_VACUO"
    if tit401 < 180.0:
        return "RISCO_SELAGEM_FRACA"
    return "CICLO_CONFORME"

petri_res = petri_bounded_check(3)
assert petri_res["bounded"] and petri_res["deadlocks"] == 0 and petri_res["mutual_exclusion"]
print("Petri (prévia):", petri_res)

In [ ]:
# ============================================================================
# 3. SIMULAÇÃO INTEGRADA DOS 4 CENÁRIOS DO MÓDULO 3
# ============================================================================

AGV_WAYPOINTS = [(1.6, 1.8), (2.2, 5.5), (3.6, 9.8), (8.2, 10.2), (13.2, 9.6)]

SCENARIOS = [
    {
        "id": "nominal",
        "label": "1. Turno Nominal",
        "hs004": False, "zs102": True, "pit301": True, "tit401": 186.0,
        "backbone_down": False, "req_prensa": False, "pos_disp": True,
        "expected_bdt": "OPERACAO_NORMAL", "expected_cart": "CICLO_CONFORME", "expected_rrt": "STANDBY"
    },
    {
        "id": "cup_shortage",
        "label": "2. Falta de Copo (ZS-102)",
        "hs004": False, "zs102": False, "pit301": True, "tit401": 186.0,
        "backbone_down": False, "req_prensa": False, "pos_disp": True,
        "expected_bdt": "ALARME_REABASTECIMENTO", "expected_cart": "PARADA_ABASTECIMENTO_ATRASO", "expected_rrt": "ROTA_ATIVA"
    },
    {
        "id": "backbone_cold",
        "label": "3. Falha Backbone RSTP + Selagem Fria",
        "hs004": False, "zs102": True, "pit301": True, "tit401": 168.0,
        "backbone_down": True, "req_prensa": True, "pos_disp": False,
        "expected_bdt": "HOLD_ESTAGIO_4", "expected_cart": "RISCO_SELAGEM_FRACA", "expected_rrt": "STANDBY"
    },
    {
        "id": "emergency",
        "label": "4. Emergência Global (HS-004)",
        "hs004": True, "zs102": False, "pit301": True, "tit401": 186.0,
        "backbone_down": False, "req_prensa": True, "pos_disp": True,
        "expected_bdt": "TRIP_EMERGENCIA", "expected_cart": "PARADA_ABASTECIMENTO_ATRASO", "expected_rrt": "ABORTO_EMERGENCIA"
    }
]

results = []
for sc in SCENARIOS:
    rstp = run_rstp(backbone_down=sc["backbone_down"])
    bdt = evaluate_bdt(hs004=sc["hs004"], req_prensa=sc["req_prensa"], tit401=sc["tit401"], pit301=sc["pit301"], pos_disp=sc["pos_disp"], zs102=sc["zs102"])
    bt = tick_behavior_tree(hs004=sc["hs004"], zs102=sc["zs102"], pit301=sc["pit301"], tit401=sc["tit401"])
    if sc["hs004"]:
        rrt_status = "ABORTO_EMERGENCIA"
    elif not sc["zs102"]:
        rrt_status = "ROTA_ATIVA"
    else:
        rrt_status = "STANDBY"
    nearest_pt, nearest_dist = kdtree_nearest_2d(AGV_WAYPOINTS, (7.5, 9.8))
    cart = cart_predict_cycle(sc["tit401"], sc["zs102"], sc["pit301"])

    assert bdt == sc["expected_bdt"], f"BDT inesperado em {sc['id']}: {bdt}"
    assert rrt_status == sc["expected_rrt"], f"RRT inesperado em {sc['id']}: {rrt_status}"
    assert cart == sc["expected_cart"], f"CART inesperado em {sc['id']}: {cart}"
    assert rstp["lost_cycles_rstp"] == 0

    results.append((sc["label"], bdt, bt["action"], rrt_status, cart, rstp["plc_cost"]))
    print(f"[OK] {sc['label']} -> BDT={bdt} | BT={bt['action']} | RRT={rrt_status} | CART={cart} | RSTP_custo={rstp['plc_cost']}")

assert len(results) == 4
print("\nTodos os asserts da Avaliação Integrada do Módulo 3 passaram com sucesso!")